## Notebook 3: Comparing Models

Where I left off: Logistic Regression caught 92% of fraud but only 6% of what it flagged was actually fraud, over 1,400 false alarms in the test set alone. Not something a real fraud review team could work with.

Here I'm trying two upgrades to see if they actually help. A Random Forest (a more flexible model that can pick up non-linear patterns), and SMOTE (a technique that generates synthetic fraud examples so the training data is more balanced). Then I'll compare all three side by side using the same fraud-class precision, recall, and F1 numbers from notebook 2, so it's a fair comparison.

### Step 5: Model Comparison

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Re-run the same preprocessing steps from notebook 2, since this is a
# fresh notebook with no memory of anything we did before
df = pd.read_csv('../data/creditcard.csv')

scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])  # scale dollar amount to mean=0, std=1
df['Time_scaled'] = scaler.fit_transform(df[['Time']])      # scale seconds-elapsed the same way
df = df.drop(['Amount', 'Time'], axis=1)  # drop the original unscaled columns

X = df.drop('Class', axis=1)  # features - everything the model is allowed to see
y = df['Class']               # target - the answer we're trying to predict (0 = normal, 1 = fraud)

# Same split as notebook 2 (same random_state=42 and stratify=y),
# so the train/test sets here are identical to the ones used for the baseline -
# this keeps the comparison fair
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### Step 5.1 - Train a Random Forest

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf_model = RandomForestClassifier(
    n_estimators=100,        # build 100 separate decision trees and combine their votes
    class_weight='balanced',  # same idea as before - pay extra attention to the rare fraud class
    random_state=42,
    n_jobs=-1                 # use all available CPU cores to train faster (trees can build in parallel)
)

# This will take noticeably longer than Logistic Regression - it's
# building 100 trees instead of one simple weighted formula
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
print(classification_report(y_test, y_pred_rf, target_names=['Normal', 'Fraud']))

              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.92      0.81      0.86        98

    accuracy                           1.00     56962
   macro avg       0.96      0.90      0.93     56962
weighted avg       1.00      1.00      1.00     56962



Logistic Regression draws one smooth mathematical boundary between fraud and normal using a weighted formula. A Random Forest works completely differently. It builds a bunch of individual decision trees (100 of them here), where each tree is basically a flowchart asking yes/no questions about the features ("is V14 less than -3? is V4 greater than 2?") to arrive at a prediction. Each tree only sees a random subset of the data and features, so no two trees end up identical, and the forest's final answer is just a majority vote across all 100.

The reason this often works better for fraud detection: decision trees can pick up on more complex, interacting patterns that a single linear formula struggles with, something like "fraud is likely if V14 is very low and V4 is high, but only when Amount is also small." That kind of conditional logic is natural for a tree, awkward for logistic regression. Combining 100 trees also helps guard against any one tree overfitting to noise.

Going in, I'm expecting precision to improve meaningfully compared to the Logistic Regression baseline (0.06 / 0.92 / 0.11), probably into the 80-90%+ range, though recall might dip a bit since Random Forest tends to be more conservative about calling something fraud.

Random Forest results (fraud class): precision 0.92, recall 0.81, F1 0.86.

| Model | Precision (Fraud) | Recall (Fraud) | F1 (Fraud) |
|---|---|---|---|
| Logistic Regression | 0.06 | 0.92 | 0.11 |
| Random Forest | 0.92 | 0.81 | 0.86 |

This basically flips the tradeoff. Instead of catching 92% of fraud while drowning the team in false alarms (94 out of every 100 flags were wrong), Random Forest gets 92% of its flags right, a massive drop in false alarms, while still catching 81% of actual fraud (about 79-80 out of 98). Doing the math, it correctly flagged roughly 79 fraud cases, missed about 19, and only misfired on around 7 normal transactions (79 / 0.92 is about 86 total flags, ~7 of them false alarms).

F1 jumping from 0.11 to 0.86 is the clearest way to see how much better balanced this model is. A review team could realistically work through 86 flagged transactions a day. 1,500+ flags from Logistic Regression just wasn't operationally workable.

The tradeoff to keep in mind: recall did drop from 92% to 81%, so more real fraud slips through than before (about 19 cases vs. ~8). Whether that's acceptable depends on the business context, and it's exactly the kind of thing threshold tuning in notebook 4 lets me adjust. I'm not locked into the model's default 50% cutoff.

### Step 5.2 - Try SMOTE

In [3]:
from imblearn.over_sampling import SMOTE

# SMOTE = Synthetic Minority Oversampling Technique.
# Instead of telling the model "pay more attention to fraud" (like class_weight did),
# SMOTE actually creates new, synthetic fraud examples by interpolating between
# real fraud cases that are similar to each other - artificially balancing the dataset
smote = SMOTE(random_state=42)

# IMPORTANT: we only apply this to the TRAINING data.
# The test set must stay 100% real, untouched data, or our evaluation
# numbers would be meaningless (we'd be testing on fake transactions)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Compare class counts before and after - this shows SMOTE actually worked
print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE: {pd.Series(y_train_smote).value_counts().to_dict()}")

# Train a fresh Random Forest on the SMOTE-balanced data.
# Note: no class_weight needed here, since SMOTE already balanced the classes directly
rf_smote = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = rf_smote.predict(X_test)
print(classification_report(y_test, y_pred_smote, target_names=['Normal', 'Fraud']))

Before SMOTE: {0: 227451, 1: 394}
After SMOTE: {0: 227451, 1: 227451}
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.82      0.82      0.82        98

    accuracy                           1.00     56962
   macro avg       0.91      0.91      0.91     56962
weighted avg       1.00      1.00      1.00     56962



class_weight='balanced' keeps the original data as-is but tells the model to weigh mistakes on fraud cases more heavily during training. SMOTE takes a different approach. It manufactures new synthetic fraud rows by picking real fraud cases that are numerically similar and generating new points between them in feature space, actually growing the training set until both classes are the same size (roughly 227,000 vs 227,000, up from 227,451 vs 394).

I'm only applying this to the training data, never the test set. The test set is supposed to represent real transactions I'm evaluating against. Injecting fake synthetic fraud into it would let the model get credit for detecting fraud that doesn't actually exist, which would make the results look better than the model would actually perform in practice.

The before/after numbers confirm the balancing worked. Training went from 227,451 normal vs. 394 fraud to an even 227,451 vs. 227,451, so SMOTE generated roughly 227,000 synthetic fraud examples.

SMOTE + Random Forest results (fraud class): precision 0.82, recall 0.82, F1 0.82.

### Step 5.3 - Full comparison table

In [4]:
# Build a simple side-by-side comparison table using the fraud-class numbers
# pulled from each classification_report printed above
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'Random Forest + SMOTE'],
    'Precision (Fraud)': [0.06, 0.92, 0.82],
    'Recall (Fraud)':    [0.92, 0.81, 0.82],
    'F1 (Fraud)':        [0.11, 0.86, 0.82],
})
print(results)

# Save this table as a CSV too - you'll want it later for the dashboard and portfolio writeup
results.to_csv('../dashboard/model_comparison.csv', index=False)

                   Model  Precision (Fraud)  Recall (Fraud)  F1 (Fraud)
0    Logistic Regression               0.06            0.92        0.11
1          Random Forest               0.92            0.81        0.86
2  Random Forest + SMOTE               0.82            0.82        0.82


| Model | Precision (Fraud) | Recall (Fraud) | F1 (Fraud) |
|---|---|---|---|
| Logistic Regression | 0.06 | 0.92 | 0.11 |
| Random Forest (class_weight) | 0.92 | 0.81 | 0.86 |
| Random Forest + SMOTE | 0.82 | 0.82 | 0.82 |

Plain Random Forest with class_weight='balanced' wins. It has both the highest precision (0.92) and the highest F1 (0.86) of the three. SMOTE actually made precision worse (0.92 down to 0.82), and only nudged recall up slightly (0.81 to 0.82).

I went in half-expecting SMOTE to win, since it's the more advanced technique on paper, but that assumption didn't hold up. My best guess for why: SMOTE's synthetic fraud examples get generated by interpolating between real fraud points, and if the real fraud cases are already fairly spread out and varied, those synthetic points can land in ambiguous territory that overlaps more with normal transactions, making the model a bit more prone to false alarms. class_weight, on the other hand, never invents new data, it just reweights what's already there, and that apparently generalizes better here.

Worth calling out in the write-up: simpler won this round. I tested the assumption instead of just defaulting to "more complex is better," and the data pushed back on it.